In [ ]:
import pandas as pd
import numpy
import os

In [ ]:
lineage_file = pd.ExcelFile(r'C:\Users\AryanKumar\Downloads\Data Lineage\Order Fulfillment Dashboard_Upstream.xlsx')
lineage_sheet = lineage_file.parse('Order Fulfillment Dashboard_Ups')
lineage_sheet

In [ ]:
df = lineage_sheet[['Name','Type','Connector','Database','Schema','Lineage Depth','Immediate upstream']].copy()
queries = df[(df['Connector'] == 'powerbi') & (df['Immediate upstream'].notnull())][['Name','Immediate upstream']].copy()
queries.rename(columns={'Name': 'Query', 'Immediate upstream': 'L1'}, inplace=True)
queries.index = pd.RangeIndex(1, len(queries)+1)
queries['L1'] = (
    queries['L1']
        .str.extract(r'\((.*?)\)')[0]
        .str.split('/', expand=True)
        .iloc[:, 3:6]
        .apply(lambda row: '.'.join(row.dropna()), axis=1)
)
queries[['Database_L1', 'Schema_L1', 'Object_L1']] = queries['L1'].str.split('.', expand=True)
queries

In [ ]:
queries_merged = queries.copy()
current_level = 2
while True:
    merge_keys = [f'Database_L{current_level-1}', f'Schema_L{current_level-1}', f'Object_L{current_level-1}']
    queries_merged = pd.merge(
        queries_merged,
        df[['Immediate upstream','Database','Schema','Name']],
        left_on=merge_keys,
        right_on=['Database', 'Schema', 'Name'],
        how='left'
    )
    queries_merged.rename(columns={'Immediate upstream': f'L{current_level}', 'Database': f'Database_L{current_level}', 'Schema': f'Schema_L{current_level}', 'Name': f'Object_L{current_level}'}, inplace=True)
    curr_col = f'L{current_level}'
    queries_merged[curr_col] = (
        queries_merged[curr_col]
            .str.split(',')
            .apply(lambda lst: [
                '.'.join(
                    item.split('(')[1]
                        .rstrip(')')
                        .split('/')[3:6]
                )
                for item in lst
            ] if isinstance(lst, list) else lst)
    ) if queries_merged[curr_col].notna().any() else queries_merged[curr_col]
    # remove any value that is the same as the previous level so we don’t loop infinitely on self-referencing objects, also handle cases where the current column is not a list
    queries_merged[curr_col] = queries_merged.apply(
        lambda r: [v for v in r[curr_col] if v not in {r[f'L{i}'] for i in range(current_level-1,0,-1)}] if isinstance(r[curr_col], list) else r[curr_col],
        axis=1
    )
    queries_merged = queries_merged.explode(curr_col)
    queries_merged.index = pd.RangeIndex(1, len(queries_merged)+1)

    queries_merged[[f'Database_L{current_level}', f'Schema_L{current_level}', f'Object_L{current_level}']] = queries_merged[curr_col].str.split('.', expand=True) if queries_merged[curr_col].notna().any() else queries_merged[[f'Database_L{current_level}', f'Schema_L{current_level}', f'Object_L{current_level}']]
    if queries_merged[curr_col].isna().all():
        queries_merged.drop(columns=[f'L{current_level}', f'Database_L{current_level}', f'Schema_L{current_level}', f'Object_L{current_level}'], inplace=True)
        print(f'No more upstream dependencies found at level {current_level}. Stopping iteration.')
        break
    print(f'Completed level {current_level}')
    current_level += 1
summarized_lineage = queries_merged.copy()[['Query'] + [col for col in queries_merged.columns if col.startswith('L')]]
try:
    print('Saving lineage to Excel file...')
    summarized_lineage.to_excel(f'C:\\Users\\AryanKumar\\Downloads\\Data Lineage\\{lineage_file.sheet_names[0]}__lineage.xlsx', index=False)
    print('Lineage saved successfully.')
except Exception as e:
    print(f'Error occurred while saving Excel file: {e}')
summarized_lineage

Completed level 2
Completed level 3
Completed level 4
Completed level 5
Completed level 6
Completed level 7
Completed level 8
Completed level 9
Completed level 10
Completed level 11
Completed level 12
Completed level 13
Completed level 14
Completed level 15
No more upstream dependencies found at level 16. Stopping iteration.
Saving lineage to Excel file...
Error occurred while saving Excel file: [Errno 13] Permission denied: 'C:\\Users\\AryanKumar\\Downloads\\Data Lineage\\Order Fulfillment Dashboard_Ups__lineage.xlsx'


,Query,L1,L2,L3,L4,L5,L6,L7,L8,L9,L10,L11,L12,L13,L14,L15
1,ACCOUNTBASE,PROD_DATALAKE.CRM_MSCRM.ACCOUNTBASE,PROD_DATALAKE.CRM_MSCRM.attrep_changes3F84AF30...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BRANCH_DIM,PROD_EDW.ENT.BRANCH_DIM,PROD_DATALAKE.CRM_MSCRM.STRINGMAPBASE,PROD_DATALAKE.CRM_MSCRM.attrep_changes3F84AF30...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BRANCH_DIM,PROD_EDW.ENT.BRANCH_DIM,PROD_DATALAKE.CRM_MSCRM.CDA_AREABASE,PROD_DATALAKE.CRM_MSCRM.attrep_changes3F84AF30...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BRANCH_DIM,PROD_EDW.ENT.BRANCH_DIM,PROD_DATALAKE.CRM_MSCRM.CDA_BRANCHBASE,PROD_DATALAKE.CRM_MSCRM.attrep_changes3F84AF30...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,BRANCH_DIM,PROD_EDW.ENT.BRANCH_DIM,PROD_DATALAKE.SALESFORCE.BRANCH__C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6408,VW_JOBORDER_DETAIL_PRICING_ANALYTICS_NAV,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_JOBORD...,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_PRICIN...,DATALAKE.PUBLIC.REPORT_FPR,DATALAKE.PUBLIC.DIM_BRAND,DATALAKE.PUBLIC.LAWSON_DL_LDG_ACACTIVITY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6409,VW_JOBORDER_DETAIL_PRICING_ANALYTICS_NAV,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_JOBORD...,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_PRICIN...,DATALAKE.PUBLIC.REPORT_FPR,DATALAKE.PUBLIC.DIM_BRAND,DATALAKE.PUBLIC.LAWSON_DL_LDG_BRAND_FORMAT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6410,VW_JOBORDER_DETAIL_PRICING_ANALYTICS_NAV,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_JOBORD...,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_PRICIN...,DATALAKE.PUBLIC.REPORT_FPR,DATALAKE.PUBLIC.DIM_ASSIGNMENT,PROD_DATALAKE.CRM_MSCRM.CDA_ASSIGNMENTBASE,PROD_DATALAKE.CRM_MSCRM.attrep_changes3F84AF30...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6411,VW_JOBORDER_DETAIL_PRICING_ANALYTICS_NAV,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_JOBORD...,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_PRICIN...,DATALAKE.PUBLIC.REPORT_FPR,DATALAKE.PUBLIC.DIM_ASSIGNMENT,PROD_DATALAKE.CRM_MSCRM.SYSTEMUSERBASE,PROD_DATALAKE.CRM_MSCRM.attrep_changes3F84AF30...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
